# Récupération des données des log du Branch and Bound beta-CROWN

In [ ]:
import re
import pandas as pd

def parse_verification_report(path):
    """
    Lit un fichier texte contenant un rapport au format donné
    et extrait les informations importantes dans un dictionnaire.
    """

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    data = {}

    # Final verified acc
    m = re.search(r"Final verified acc:\s*([\d\.]+)%", text)
    if m:
        data["final_verified_accuracy"] = float(m.group(1))

    # Problem instances count
    m = re.search(r"Problem instances count:\s*(\d+)", text)
    if m:
        data["problem_instances_count"] = int(m.group(1))

    # total verified, total falsified, timeout
    m = re.search(
        r"total verified.*?:\s*(\d+)\s*,\s*total falsified.*?:\s*(\d+)\s*,\s*timeout:\s*(\d+)",
        text
    )
    if m:
        data["verified"] = int(m.group(1))
        data["falsified"] = int(m.group(2))
        data["timeout"] = int(m.group(3))

    # mean / max time for all instances
    m = re.search(
        r"mean time for ALL instances.*?:([\d\.]+).*?max time:\s*([\d\.]+)",
        text
    )
    if m:
        data["mean_time_all"] = float(m.group(1))
        data["max_time_all"] = float(m.group(2))

    # mean / max time for verified SAFE
    m = re.search(
        r"mean time for verified SAFE instances.*?:\s*([\d\.]+).*?max time:\s*([\d\.]+)",
        text
    )
    if m:
        data["mean_time_safe"] = float(m.group(1))
        data["max_time_safe"] = float(m.group(2))

    # safe-incomplete list
    m = re.search(r"safe-incomplete.*?index:\s*\[(.*?)\]", text)
    if m:
        indices = m.group(1).strip()
        data["safe_incomplete_indices"] = (
            [] if indices == "" else [int(x) for x in indices.split(",")]
        )

    return data


In [ ]:
result = parse_verification_report("/share/homes/boyerma/FastSDPCertification/results/benchmark/6x100-0.026/Branch-and-Bound-beta-CROWN/data_index=0-71.txt")
print(result)


In [ ]:
import os
import sys

folder = "/share/homes/boyerma/FastSDPCertification/results/benchmark/6x100-0.026/Branch-and-Bound-beta-CROWN/" 
results = pd.DataFrame(columns=[
    "filename","final_verified_accuracy","problem_instances_count",
    "verified","falsified","timeout",
    "mean_time_all","max_time_all",
    "mean_time_safe","max_time_safe",
    "safe_incomplete_indices"
])

for filename in os.listdir(folder):
    if filename.endswith(".txt"):
        path = os.path.join(folder, filename)
        result = parse_verification_report(path)
        print(f"Results for {filename}:")
        print(result)
        results = pd.concat([results, pd.DataFrame([{"filename": filename, **result}])], ignore_index=True)
        print("-" * 40)

In [ ]:
results["total"] = results["verified"] + results["falsified"] + results["timeout"]

In [ ]:
results["verified"].sum()

In [ ]:
results["falsified"].sum()

In [ ]:
results.head(20)

In [ ]:
results["total_time"] = results["mean_time_all"] * results["total"]

In [ ]:
results["total_time"].mean()

# Verification output

In [ ]:
import pickle

with open('/share/homes/boyerma/alpha-beta-CROWN/complete_verifier/verification_output.pkl', 'rb') as f:
    verification_output = pickle.load(f)
    


In [ ]:
verification_output

# Vérification des exemples adverses

In [1]:
#!/usr/bin/env python3
"""
Script to extract X and Y tensors from counterexample file.
Reads the cex_path file and extracts:
- X tensor: 784 values (X_0 to X_783)
- Y tensor: 10 values (Y_0 to Y_9)
"""

import re
import numpy as np
from pathlib import Path


def extract_tensors_from_cex(cex_path):
    """
    Extract X and Y tensors from counterexample file.
    
    Args:
        cex_path (str): Path to the counterexample file
        
    Returns:
        tuple: (X_tensor, Y_tensor) as numpy arrays
    """
    X_values = {}
    Y_values = {}
    
    with open(cex_path, 'r') as f:
        content = f.read()
    
    # Parse all (name value) pairs
    pattern = r'\((\w+_\d+)\s+([-\d.]+)\)'
    matches = re.findall(pattern, content)
    
    for name, value in matches:
        value = float(value)
        
        if name.startswith('X_'):
            idx = int(name.split('_')[1])
            X_values[idx] = value
        elif name.startswith('Y_'):
            idx = int(name.split('_')[1])
            Y_values[idx] = value
    
    # Convert to numpy arrays (sorted by index)
    X_tensor = np.array([X_values[i] for i in sorted(X_values.keys())])
    Y_tensor = np.array([Y_values[i] for i in sorted(Y_values.keys())])
    
    return X_tensor, Y_tensor


In [2]:
"""Main function."""
cex_path = '/share/homes/boyerma/alpha-beta-CROWN/complete_verifier/test_margot.txt'

# Check if file exists
if not Path(cex_path).exists():
    print(f"Error: File '{cex_path}' not found")
    exit()
    
# Extract tensors
X_adv_bb, Y_adv_bb = extract_tensors_from_cex(cex_path)

print(f"Successfully extracted tensors from '{cex_path}'")
print(f"\nX tensor shape: {X_adv_bb.shape}")
print(f"X tensor min: {X_adv_bb.min():.6f}, max: {X_adv_bb.max():.6f}, mean: {X_adv_bb.mean():.6f}")
print(f"X tensor first 10 values: {X_adv_bb[:10]}")
print(f"X tensor last 10 values: {X_adv_bb[-10:]}")

print(f"\nY tensor shape: {Y_adv_bb.shape}")
print(f"Y tensor values:\n{Y_adv_bb}") 


Successfully extracted tensors from '/share/homes/boyerma/alpha-beta-CROWN/complete_verifier/test_margot.txt'

X tensor shape: (784,)
X tensor min: 0.000000, max: 0.974000, mean: 0.113157
X tensor first 10 values: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
X tensor last 10 values: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Y tensor shape: (10,)
Y tensor values:
[-10.21920776  -7.33659267  -3.67729521   1.17926931   8.22635174
   0.35640112  -6.07050467   2.92742085   2.75901294   6.85418177]


In [4]:
import torch
from adversarial_attacks import PGDAttack
from data import load_dataset
from tools import get_project_path
from torch.utils.data import DataLoader
from networks import ReLUNN

dataset = load_dataset(get_project_path(f"config/mnist-6x100.yaml"))
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

network = ReLUNN.from_yaml(get_project_path("config/mnist-6x100.yaml"))
epsilon = 0.026

device_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i, (x, ytrue) in enumerate(dataloader):
    if i not in [97]:
    
        continue
    
    print(f"Example {i}:")
    print(f"x: {x}")
    print(f"ytrue: {ytrue}")

    diff = x.flatten() - X_adv_bb
    print(f"diff min : {diff.min():.6f}, max: {diff.max():.6f}, mean: {diff.mean():.6f}")
    
    X_adv_bb = torch.tensor(X_adv_bb.reshape(1,784), dtype = torch.float)

    network.forward(X_adv_bb)

    pgd_attack = PGDAttack(network, eps=epsilon, norm="inf")
    with torch.enable_grad():
        network.to(device_)
        x = x.to(device_)
        ytrue = ytrue.to(device_)
        adv_inputs = pgd_attack.forward(x, ytrue)
        print("STUDY : adv_inputs:", adv_inputs)
        print("STUDY:  network prediction on adv_inputs:", network(adv_inputs))
        y_pred_adv = torch.argmax(network(adv_inputs), dim=0)
        print("STUDY : y_pred_adv:", y_pred_adv)
        diff = x - adv_inputs
        print("STUDY : diff between x and adv_inputs:", diff.max(), diff.min())

        values_layer = network.return_values_each_layer(adv_inputs)
        print("values_layer in certification : ", values_layer)
        
        


config :  {'data': {'name': 'mnist', 'path': 'data/datasets/mnist_subset_10_per_class.pth', 'num_classes': 10, 'num_samples': 100}, 'input_ball': {'norm': 'Linf', 'epsilon': 0.026}, 'network': {'name': '6x100', 'path': 'data/models/mnist_adv_6x100.pt', 'K': 7, 'n': [784, 100, 100, 100, 100, 100, 100, 10]}, 'models': [{'certification_model_name': 'MdSDP', 'cuts': ['RLT', 'triangularization', 'Tij', 'beta_logits_comparaison_1', 'beta_logits_comparaison_2', 'McC_betaz_logits', 'Tij_before_penultimate_layer'], 'RLT_props': [1.0], 'all_combinations_cuts': False, 'MATRIX_BY_LAYERS': True, 'LAST_LAYER': False, 'use_fusion': False, 'use_callback': False, 'use_active_neurons': False, 'use_inactive_neurons': False, 'keep_penultimate_actives': True, 'bounds_method': 'alpha-CROWN', 'alpha_1': 0.5, 'alpha_2': 0.5}]}
file :  /share/homes/boyerma/FastSDPCertification/config/mnist-6x100.yaml
K :  7
n :  [784, 100, 100, 100, 100, 100, 100, 10]
parametres :  OrderedDict([('layers.Layer_1_Linear.weight',

NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
NOT TARGETED ATTACK
STUDY : adv_inputs: tensor([[[[0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 0.0260,
           0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 0.0260,
           0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 0.0260,
           0.0260, 0.0260, 0.0260, 0.0260],
          [0.0260, 0.0260, 0.0260, 0.0260, 0.0260, 

/tmp/ipykernel_80540/3202026509.py:25: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  diff = x.flatten() - X_adv_bb
